# LLM · Lecture 13 — Agentic Code Execution & the Self-Debugging Loop

Give an agent the ability to **write and run code** — safely — so it can compute exactly and **fix its own mistakes**. This notebook runs the whole lecture: why a model needs code, the **generate → run → read → fix** loop (both a deterministic stubbed version and a live one on the class model), the sandbox controls that make running untrusted code survivable, the **over-restriction** trap, test-driven self-debug with **hidden tests**, and an **escalation** policy for when the loop won't converge.

**Two kinds of cell.** The self-debugging *mechanics* (the stubbed loop, over-restriction, escalation, the float-bug demo) are pure stdlib and **always run**. The parts that call the model (`gemma-4-E4B-it` via the class proxy) are **key-guarded** — paste your course API key to fire them.

> ⚠️ Everything here `exec()`s code **in this notebook's own process**. That's fine for a teaching demo with code *we* wrote, but it is **not a sandbox**. §3 explains what a real one needs, and why you must never `exec()` untrusted model output on your own host.

## 0. Setup

In [ ]:
import os, re, traceback

os.environ.setdefault('OPENAI_BASE_URL', 'https://llm.nat-d.uk/v1')   # the class proxy
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # <-- paste your key (portal: Profile -> API key), then RE-RUN this cell
MODEL = 'gemma-4-E4B-it'
HAVE_KEY = bool(os.environ.get('OPENAI_API_KEY'))

def client():
    from openai import OpenAI          # preinstalled in Colab
    return OpenAI()                    # reads OPENAI_API_KEY + OPENAI_BASE_URL

def need_key():
    print('No OPENAI_API_KEY set — set it in the cell above to run the live (model) cells.')
    print('(The stubbed / stdlib cells below run regardless.)')

print('base:', os.environ['OPENAI_BASE_URL'], '| model:', MODEL, '| key set:', HAVE_KEY)

## 1. Why give an agent code?

LLMs are next-token predictors: great at *describing* a computation, unreliable at *performing* one. The fix is to let the model **write code and run it** — a Python interpreter computes the exact value instead of guessing. See the gap directly (no key needed):

In [ ]:
# The kind of arithmetic a model pattern-matches (and often gets subtly wrong)...
expr = "198473 * 7711 + 20250929"                  # OUR literal, not model output -> safe to eval
print("code computes it exactly:", eval(expr))     # ground truth, every time
print("...a model would have to *guess* this digit-by-digit — that's what running code fixes.")

Once an agent can run code, a whole class of tasks opens up — real arithmetic and data analysis, file/text transforms, and most importantly **self-correction**: run the code, see it fail, and fix it against an objective signal. That last one is the heart of this lecture.

## 2. Executing model output is dangerous

The moment you run text the model produced, you're executing **code you didn't write and can't fully predict** — and it may have been *manipulated* (a prompt injection hidden in a file or web page). Untrusted generated code could delete files (`shutil.rmtree`), read and **exfiltrate** credentials, call an attacker's server, or spin forever.

> **Never `exec()` model output on your own host.** A run must be *survivable* — at worst you throw away a container, not your laptop. The defence is a **sandbox**.

## 3. Anatomy of a sandbox

A good sandbox layers independent controls; map each to the **specific attack it stops**:

| Control | Stops |
|---|---|
| **Isolation** (throwaway container / gVisor / microVM, no host FS) | `shutil.rmtree('/')`, reading other users' data |
| **Time limit** (e.g. 30s wall-clock) | `while True: pass`, fork bombs |
| **Memory limit** (e.g. 256 MiB) | `x = [0] * 10**12` exhausting the host |
| **Network off** | exfiltration + download-and-run of a payload |
| **No secrets inside** | nothing valuable to steal even if code runs |
| **Output cap** | log-flood / disk-fill |

Two nuances the lecture stresses: **restricting `__builtins__` is *not* a security boundary** (a determined payload escapes in-process guards — real isolation is the OS/container layer), and **network-off + no-secrets-inside does most of the practical work**. You've used a real sandbox all term: **Piston** (the Run button + Scratch page) runs each submission as a fresh, throwaway, network-restricted process — never on your laptop.

**The isolation ladder** — pick by *how hostile your inputs are*, trading strength for startup latency:

| Rung | Mechanism | Stops | Use when |
|---|---|---|---|
| **Separate process** | OS process + `rlimit`, dropped privileges | accidental host corruption (*not* a kernel exploit) | semi-trusted inputs (your own model/prompts) |
| **OS container** (Docker) | Linux namespaces + cgroups, *shared kernel* | filesystem/network reach if configured tightly | default for most coding agents; pair with network-off |
| **gVisor** | user-space kernel intercepts syscalls | kernel-level escapes | genuinely untrusted code (public users, injected inputs) |
| **microVM** (Firecracker) | hardware virtualisation, own kernel | almost everything short of a hypervisor bug | multi-tenant, hostile-by-assumption |

No matter the rung, **network-off + no-secrets-inside does most of the practical work**: with nowhere to exfiltrate and nothing worth stealing, even a kernel escape has a tiny payoff. Piston sits around the container rung with a cluster-enforced network allow-list — strong enough for class inputs.

## 4. Tools for a coding agent

A coding assistant is mostly three tools (Lecture 10 shape — your code runs them, the output becomes the next context): **`read_file(path)`** (inspect code/spec/a failing test), **`run(code)`** (execute in the sandbox → stdout/stderr), and **`run_tests()`** (the **pass/fail** signal that tells the loop when to stop). `read_file`/`run` are one-liners that forward to the sandbox (never local, e.g. `def run(code): return sandbox_execute(code)`); here we define the one this notebook drives — `run_tests`:

In [ ]:
def run_tests(code, tests, safe_builtins):
    """Exec `code`, then check each (call, expected). Raise AssertionError on mismatch.
    NOTE: real safety is the sandbox; the builtins dict is a convenience guard, not a boundary."""
    ns = {}
    exec(code, {"__builtins__": safe_builtins}, ns)   # toy guard; real isolation = Piston/container
    for call, expected in tests:
        got = eval(call, ns)
        assert got == expected, f"{call} -> {got!r}, expected {expected!r}"

# Allow the builtins a reasonable solution legitimately needs (see §7 for WHY this matters).
SAFE_BUILTINS = {
    "range": range, "len": len, "list": list, "int": int, "float": float,
    "round": round, "abs": abs, "min": min, "max": max, "sum": sum, "sorted": sorted,
}
print("run_tests + SAFE_BUILTINS ready")

**Two ways to wire the model in.** The §5–§6 loop uses the simpler **reply-with-code convention** — the model just replies with code in a fenced block and your loop runs it — which is right when the model's only job each turn is "write the function." When the model must **orchestrate several tools itself** (read a file, then run, then test, choosing which and when), register `run` as a **native Lecture-10 tool** instead (a function + a JSON schema), pass `tools=[run_tool]`, and dispatch the `tool_calls`. Rule of thumb: *one fixed step per turn → reply-with-code; the model orchestrates many tools → native `tools=`* (that's the HW7 extension, P5).

## 5. The self-debugging loop, stubbed — trace every step (always runs)

The heart of a coding assistant: **generate → run → read → fix**, repeated until the tests pass or a **retry cap** stops it. To see the mechanics with *zero variance*, we first replace the model with a scripted stub that hands back canned attempts. The loop can't tell — it still does all four phases. Attempt 1 is the textbook round-half-up trick with a real **floating-point bug**; attempt 2 is the integer fix.

In [ ]:
MAX_ATTEMPTS = 3                       # cap retries -- never loop forever

SCRIPTED = [
    "def pct(part, whole):\n    return int(part / whole * 100 + 0.5)",    # buggy (float)
    "def pct(part, whole):\n    return int((part * 100 / whole) + 0.5)",  # fixed (integer arithmetic)
]
TESTS = [("pct(1, 8)", 13), ("pct(23, 40)", 58)]   # 2nd is the FP trap

class StubModel:
    def __init__(self, replies): self.replies, self.i = replies, 0
    def reply(self, _messages):                    # ignores convo; next scripted reply
        r = self.replies[min(self.i, len(self.replies) - 1)]; self.i += 1
        return r

stub = StubModel(SCRIPTED)
messages = [{"role": "user", "content": "Write pct(part, whole)."}]
for attempt in range(1, MAX_ATTEMPTS + 1):
    code = stub.reply(messages)
    messages.append({"role": "assistant", "content": code})
    print(f"--- attempt {attempt} ---\n{code}")
    try:
        run_tests(code, TESTS, SAFE_BUILTINS)      # RUN + the pass/fail signal
        print("All tests passed.")
        break
    except Exception:
        err = traceback.format_exc()               # READ: the full traceback...
        print("FAILED:", err.strip().splitlines()[-1])
        messages.append({"role": "user",           # FIX: ...feed it back to the model
            "content": f"That failed:\n{err}\nFix the function."})
else:
    print(f"Gave up after {MAX_ATTEMPTS} attempts.")

Trace it: attempt 1 fails `pct(23, 40) -> 57, expected 58`; that traceback is appended to the conversation; attempt 2 multiplies *before* dividing (`2300 / 40 == 57.5` exactly — order-preserving; the purely-integer `(part*200 + whole)//(whole*2)` works too) and passes. **The traceback is the gold** — models fix code far better when they see exactly *how* it failed. And notice the conversation *grows*: each failed attempt appends the broken code + error, so the model accumulates context about what didn't work.

Two rules of loop hygiene the toy glosses over: (1) **trust the tests, not the model's word** — "it works now" means the suite is green, not that the model *claimed* success. (2) That growing history **balloons the prompt and the bill**; in production, *truncate or summarise* old attempts — keep the latest failing code + error, drop the rest.

### Why the float bug? (always runs)
The buggy trick reads `part/whole*100` in float, which lands *just below* the true value:

In [ ]:
print("23/40*100 =", repr(23/40*100), "-> + 0.5 =", repr(23/40*100 + 0.5), "-> int() =", int(23/40*100 + 0.5))
print("2300/40   =", repr(2300/40),   "  (integer-first arithmetic is exact) -> int(+0.5) =", int(2300/40 + 0.5))

## 6. The live self-debugging loop (needs a key)

Now swap the stub for the real model and watch it **re-derive** the same fix from the same traceback. Same loop; only `ask()` changed. The spec is chosen so attempt 1 fails on a genuine logic bug, not a sandbox artefact. (It's a 4B model at temp 0 — it *usually* shows the buggy attempt then the fix, but may fix it on attempt 1, or rarely not converge.)

In [ ]:
SPEC = ("Write a function `pct(part, whole)` that returns what percentage `part` is of `whole`, "
        "rounded to the NEAREST integer, where a value ending in exactly .5 rounds UP. "
        "Example: pct(1, 8) is 12.5%, which rounds to 13. Assume whole > 0.")
LIVE_TESTS = [("pct(1, 8)", 13), ("pct(3, 8)", 38), ("pct(23, 40)", 58), ("pct(29, 200)", 15)]

def extract_code(text):
    if "```" in text:
        block = text.split("```")[1]
        for tag in ("python", "py"):
            if block.startswith(tag): block = block[len(tag):]; break
        return block.strip()
    return text.strip()

def looks_like_code(text):
    try: compile(text, "<model>", "exec"); return True     # never exec() prose blindly
    except SyntaxError: return False

def ask(messages):
    return client().chat.completions.create(model=MODEL, temperature=0, messages=messages
                                            ).choices[0].message.content

def self_debug(spec, tests, max_attempts=3):
    messages = [
        {"role": "system", "content": "You are a terse Python coding assistant. Use ONLY builtins, "
            "no imports, no comments. Reply with ONLY the function code in a fenced block."},
        {"role": "user", "content": spec},
    ]
    for attempt in range(1, max_attempts + 1):
        raw = ask(messages); code = extract_code(raw)
        messages.append({"role": "assistant", "content": raw})
        if not looks_like_code(code):                       # handle "no code" (prose)
            print(f"--- attempt {attempt}: no runnable code, re-asking ---")
            messages.append({"role": "user", "content": "Return ONLY the function code in a fenced ```python block."})
            continue
        print(f"--- attempt {attempt} ---\n{code}")
        try:
            run_tests(code, tests, SAFE_BUILTINS)
            print("All tests passed."); return True
        except Exception:
            err = traceback.format_exc()
            print("FAILED:", err.strip().splitlines()[-1])
            messages.append({"role": "user",
                "content": f"That failed:\n{err}\nFix the function. Reply with ONLY the code in a fenced block."})
    print(f"Gave up after {max_attempts} attempts."); return False

if HAVE_KEY:
    self_debug(SPEC, LIVE_TESTS)
else:
    need_key()

## 7. The over-restriction trap — a guard that *invents* bugs (always runs)

Restricting `__builtins__` *feels* safe, but a namespace too small produces **spurious `NameError`s** — and the model burns retries "fixing" a problem your sandbox invented, while the real spec drifts. Watch it happen on a *correct* answer:

In [ ]:
TOO_SMALL = {"range": range, "len": len}          # no max, no sorted, no int...
good_code = "def top(xs):\n    return max(xs)"     # a perfectly correct answer
try:
    ns = {}; exec(good_code, {"__builtins__": TOO_SMALL}, ns); print(ns["top"]([3, 9, 2]))
except NameError as e:
    print("spurious:", e, "  <- the code is CORRECT; the guard invented this error")

ns = {}; exec(good_code, {"__builtins__": {"max": max}}, ns)   # allow what the task needs
print("with max allowed:", ns["top"]([3, 9, 2]), " <- same code, now works")

Fed that `NameError`, the model would rewrite `max(xs)` as a manual loop — solving a non-problem. And a short builtins list buys **almost no safety** anyway (§3: it's not a boundary). Keep the list generous; lean on the real sandbox.

## 8. Test-driven self-debug: visible vs hidden tests (always runs)

A disciplined pattern: write the asserts *first* (the contract) and let the model chase them — but **hold some tests back**. A model that special-cases the *visible* tests gets caught by the *hidden* ones. Show the buggy attempt-1 solution passes the visible suite yet fails hidden — the exact gap the visible suite alone would ship:

In [ ]:
VISIBLE = [("pct(1, 8)", 13), ("pct(3, 8)", 38)]      # shown to the model
HIDDEN  = [("pct(23, 40)", 58), ("pct(29, 200)", 15)] # never shown; run last

buggy = "def pct(part, whole):\n    return int(part / whole * 100 + 0.5)"
def check(code, cases):
    try: run_tests(code, cases, SAFE_BUILTINS); return "PASS"
    except AssertionError as e: return f"FAIL ({e})"

print("buggy vs VISIBLE:", check(buggy, VISIBLE), " <- visible suite alone says 'done'")
print("buggy vs HIDDEN :", check(buggy, HIDDEN),  " <- the float bug the visible suite MISSED")

## 9. When the loop won't converge: escalate, don't spin (always runs)

The retry cap bounds *cost*, not *success* — non-convergence is a **first-class outcome**. When the cap is hit, escalate deliberately (cheapest lever first: raise temperature → add a hint → stronger model → simplify → **human**). Encode it as a *policy function*, not a tangle of `if`s, and make the terminal rung a clean human hand-off:

In [ ]:
def escalate(attempt, base_temp=0.0):
    # Return (temperature, model, give_up) for this attempt's retry.
    if attempt <= 2: return (base_temp, MODEL, False)   # cheap, deterministic first
    if attempt <= 4: return (0.7, MODEL, False)         # diversify before upgrading
    return (0.7, MODEL, True)                           # cap hit -> escalate to a human

for attempt in range(1, 6):
    temp, model, give_up = escalate(attempt)
    if give_up:
        print("Escalating to human: last code + traceback handed off.")   # must lead somewhere
        break
    print(f"attempt {attempt}: model={model} temp={temp}")   # ... call + run + read + fix

## 10. Practice

**P1 — a different rounding rule.** Change `SPEC` to round `.5` *down* (toward zero) and confirm the live loop re-derives a fix (needs a key). Offline, predict what the *stubbed* attempt-1 would now get wrong.

In [ ]:
import math
# TODO: with a key, edit SPEC to "round .5 DOWN (toward zero)" and re-run self_debug(...).
# Offline sanity: pct(1,8) is exactly 12.5 -> half-UP gives 13, half-DOWN gives 12.
print("round-half-up   pct(1,8) ->", math.floor(1/8*100 + 0.5), " (13)")
print("round-half-down pct(1,8) ->", math.ceil(1/8*100 - 0.5),  " (12 — .5 goes the other way)")

**P2 — add a hidden edge case.** Add `("pct(0, 5)", 0)` to `HIDDEN` and confirm both the buggy and fixed versions pass it (a *regression guard*). Then add a case the buggy one gets wrong and watch which suite catches it. (Always runs.)

In [ ]:
fixed = "def pct(part, whole):\n    return int((part * 100 / whole) + 0.5)"
guard_case = [("pct(0, 5)", 0)]
print("buggy vs guard:", check(buggy, guard_case), "| fixed vs guard:", check(fixed, guard_case))
# TODO: add a case the buggy version fails, then compare VISIBLE vs HIDDEN reporting.

**P3 — MAX_ATTEMPTS = 1.** Re-run the §5 stubbed loop with `MAX_ATTEMPTS = 1` and see the `for/else` fire — the failure surfaces cleanly to a human instead of looping. (Always runs.)

In [ ]:
stub = StubModel(SCRIPTED)                  # reset the stub
messages = [{"role": "user", "content": "Write pct(part, whole)."}]
for attempt in range(1, 1 + 1):             # MAX_ATTEMPTS = 1
    code = stub.reply(messages)
    try:
        run_tests(code, TESTS, SAFE_BUILTINS); print("passed"); break
    except Exception:
        print("attempt", attempt, "FAILED:", traceback.format_exc().strip().splitlines()[-1])
else:
    print("Gave up after 1 attempts.  <- the human-visible failure path")

**P4 (paper).** List the resource limits a backend runner sets (timeout, memory, rate limit) and the attack each stops; then name the two controls that come from the **platform** instead (Piston's throwaway process + no host FS; the cluster's proxy-only network allow-list). **P5 — HW7 on-ramp:** register `run(code)` and `run_tests()` as native Lecture-10 tools and let the model drive the read → run → test cycle itself, your loop enforcing the sandbox + step cap + clean give-up.

## Key terms
- **Code execution / interpreter** — letting an agent run code it writes, to compute exactly and check its own work.
- **Sandbox** — isolated env (container / gVisor / microVM) with time, memory, network limits.
- **Self-debugging loop** — generate → run → **read error** → fix → repeat, to a retry cap.
- **Traceback feedback** — feeding the exact error back so the model fixes the specific failure.
- **Retry cap** — a hard iteration limit so the agent can't run forever.
- **Hidden test** — an assertion the model never sees, run last, to catch overfitting.
- **Escalation policy** — raise temp → hint → stronger model → simplify → human, when the cap is hit.
- **Piston** — the isolated executor behind the Run button + Scratch page all term.